In [ ]:
"""
EV Guided Charging Lane — Site Selection Analysis
Corridors: STL → Chicago (I-55/I-57) | STL → Kansas City (I-70)

Hard Requirements per segment:
  - Room for a new dedicated lane (median width / ROE available)
  - Speed limit zone supports 60 mph down to 30 mph transition
  - >= 10 miles of open, straight road
  - No environmental constraints (wetlands / protected areas)

Scoring Dimensions (weighted composite):
  - EV traffic volume (EV share of AADT)
  - Power infrastructure proximity (substations / transmission)
  - Weather risk (lower = better; northern IL segments penalized)
  - Incident rate (lower = better; urban segments penalized)
  - Interchange density (fewer = better for guided lane entry/exit)

Data Sources (Snowflake tables — update connection string below):
  ROAD_SEGMENTS (103)   — 10-mile segments with GEOGRAPHY line strings
  TRAFFIC_COUNTS (103)  — AADT with 2-12% EV share, 15-35% trucks
  POWER_INFRA (55)      — Substations (30) + transmission lines (25)
  INTERCHANGES (343)    — Interchange locations
  ENV_CONSTRAINTS (17)  — Wetlands and protected areas as polygons
  WEATHER_RISK (103)    — Risk scores per segment
  INCIDENTS (103)       — Crash rates per segment
"""

import os
import sys

# ── Optional: connect to Snowflake ───────────────────────────────────────────
# Uncomment and fill in credentials to run against your real tables.
# pip install snowflake-connector-python pandas tabulate
#
# import snowflake.connector
# conn = snowflake.connector.connect(
#     user='YOUR_USER',
#     password='YOUR_PASSWORD',
#     account='YOUR_ACCOUNT',
#     warehouse='YOUR_WAREHOUSE',
#     database='YOUR_DATABASE',
#     schema='YOUR_SCHEMA',
# )

# ── Dependencies ──────────────────────────────────────────────────────────────
try:
    import pandas as pd
except ImportError:
    print("pandas not found. Install with: pip install pandas tabulate")
    sys.exit(1)

# ── Configuration ─────────────────────────────────────────────────────────────
WEIGHTS = {
    "ev_traffic":        0.25,
    "power_proximity":   0.25,
    "low_weather_risk":  0.20,
    "low_incident_rate": 0.15,
    "low_interchange":   0.15,
}

THRESHOLDS = {
    "min_ev_share_pct":    4.0,   # Minimum EV % of AADT
    "max_interchange_count": 4,   # Max interchanges within segment
    "max_power_dist_mi":   5.0,   # Max miles to nearest substation/line
    "grade_cutoff":        "AB",  # "A", "AB", or "ABC"
}

GRADE_THRESHOLDS = {"A": 0.68, "B": 0.50}  # composite score cutoffs


# ─────────────────────────────────────────────────────────────────────────────
# OPTION A: Load from Snowflake
# Replace this function body with a real query when connected.
# ─────────────────────────────────────────────────────────────────────────────
def load_from_snowflake(conn) -> pd.DataFrame:
    """
    Joins all 7 source tables into a flat segment-level DataFrame.
    Returns one row per ROAD_SEGMENT with all scoring columns.
    """
    query = """
        SELECT
            rs.segment_id,
            rs.corridor,
            rs.route_name,
            rs.mile_marker_start,
            rs.mile_marker_end,
            ST_ASTEXT(rs.geog)          AS geometry_wkt,

            -- Traffic
            tc.aadt,
            tc.ev_share_pct,
            tc.truck_pct,

            -- Infrastructure
            pi_agg.min_dist_mi          AS power_dist_mi,
            pi_agg.nearest_source_id    AS substation_id,

            -- Interchanges
            ic_agg.interchange_count,

            -- Environmental
            CASE WHEN ec.segment_id IS NOT NULL THEN TRUE ELSE FALSE END AS env_constraint,

            -- Weather
            wr.risk_score               AS weather_risk,

            -- Incidents
            inc.crash_rate_normalized   AS incident_rate,

            -- Hard-requirement flags (set these in your source or derive here)
            rs.has_straight_10mi,
            rs.has_lane_room,
            rs.speed_zone_ok,
            rs.median_width_ft,
            rs.roe_available

        FROM ROAD_SEGMENTS rs

        JOIN TRAFFIC_COUNTS tc
            ON rs.segment_id = tc.segment_id

        -- Nearest power source distance
        LEFT JOIN (
            SELECT
                rs2.segment_id,
                MIN(ST_DISTANCE(rs2.geog, pi.geog) / 1609.34) AS min_dist_mi,
                FIRST_VALUE(pi.source_id) OVER (
                    PARTITION BY rs2.segment_id
                    ORDER BY ST_DISTANCE(rs2.geog, pi.geog)
                ) AS nearest_source_id
            FROM ROAD_SEGMENTS rs2
            JOIN POWER_INFRA pi ON TRUE
            GROUP BY rs2.segment_id
        ) pi_agg ON rs.segment_id = pi_agg.segment_id

        -- Interchange count per segment
        LEFT JOIN (
            SELECT segment_id, COUNT(*) AS interchange_count
            FROM INTERCHANGES
            GROUP BY segment_id
        ) ic_agg ON rs.segment_id = ic_agg.segment_id

        -- Environmental constraint overlap
        LEFT JOIN (
            SELECT DISTINCT rs3.segment_id
            FROM ROAD_SEGMENTS rs3
            JOIN ENV_CONSTRAINTS ec
                ON ST_INTERSECTS(rs3.geog, ec.geog)
        ) ec ON rs.segment_id = ec.segment_id

        JOIN WEATHER_RISK wr
            ON rs.segment_id = wr.segment_id

        JOIN INCIDENTS inc
            ON rs.segment_id = inc.segment_id

        ORDER BY rs.corridor, rs.mile_marker_start
    """
    cursor = conn.cursor()
    cursor.execute(query)
    df = cursor.fetch_pandas_all()
    df.columns = [c.lower() for c in df.columns]
    return df


# ─────────────────────────────────────────────────────────────────────────────
# OPTION B: Synthetic data (runs without Snowflake)
# Mirrors the schema of load_from_snowflake() exactly.
# ─────────────────────────────────────────────────────────────────────────────
def load_synthetic_data() -> pd.DataFrame:
    import random
    random.seed(42)

    def r(): return random.random()

    rows = []

    # STL–Chicago: 55 segments (I-55/I-57)
    for i in range(55):
        near_metro = i < 5 or i > 48
        in_northern_il = i > 35
        rows.append({
            "segment_id":         f"STL-CHI-{i+1:03d}",
            "corridor":           "stl-chi",
            "route_name":         "I-55 / I-57",
            "mile_marker_start":  10 + i * 5,
            "mile_marker_end":    20 + i * 5,
            "geometry_wkt":       f"LINESTRING(-90.19 38.63, {-90.19 + (i/54)*2.56:.4f} {38.63 + (i/54)*3.25:.4f})",
            "aadt":               int(18000 + r() * 32000),
            "ev_share_pct":       round(2 + r() * 10, 1),
            "truck_pct":          round(15 + r() * 20, 1),
            "power_dist_mi":      round(0.2 + r() * 8, 1),
            "substation_id":      f"SUB-{int(r()*30+1):03d}" if r() > 0.5 else None,
            "interchange_count":  int(3 + r() * 5) if near_metro else int(r() * 4),
            "env_constraint":     r() < (0.22 if 30 < i < 50 else 0.12),
            "weather_risk":       round(0.4 + r() * 0.5, 2) if in_northern_il else round(0.1 + r() * 0.35, 2),
            "incident_rate":      round(0.45 + r() * 0.5, 2) if near_metro else round(0.05 + r() * 0.35, 2),
            "has_straight_10mi":  r() > (0.6 if near_metro else 0.3),
            "has_lane_room":      r() > (0.55 if near_metro else 0.25),
            "speed_zone_ok":      not near_metro or r() > 0.4,
            "median_width_ft":    int(20 + r() * 40),
            "roe_available":      r() > (0.55 if near_metro else 0.3),
        })

    # STL–Kansas City: 48 segments (I-70)
    for i in range(48):
        near_metro = i < 5 or i > 42
        rows.append({
            "segment_id":         f"STL-KC-{i+1:03d}",
            "corridor":           "stl-kc",
            "route_name":         "I-70",
            "mile_marker_start":  10 + i * 5,
            "mile_marker_end":    20 + i * 5,
            "geometry_wkt":       f"LINESTRING(-90.19 38.63, {-90.19 - (i/47)*4.39:.4f} {38.63 + (i/47)*0.47:.4f})",
            "aadt":               int(15000 + r() * 28000),
            "ev_share_pct":       round(2 + r() * 10, 1),
            "truck_pct":          round(18 + r() * 17, 1),
            "power_dist_mi":      round(0.3 + r() * 9, 1),
            "substation_id":      f"SUB-{int(r()*30+1):03d}" if r() > 0.45 else None,
            "interchange_count":  int(3 + r() * 5) if near_metro else int(r() * 3),
            "env_constraint":     r() < (0.18 if 15 < i < 30 else 0.1),
            "weather_risk":       round(0.08 + r() * 0.32, 2),
            "incident_rate":      round(0.4 + r() * 0.5, 2) if near_metro else round(0.05 + r() * 0.3, 2),
            "has_straight_10mi":  r() > (0.55 if near_metro else 0.2),
            "has_lane_room":      r() > (0.5 if near_metro else 0.2),
            "speed_zone_ok":      not near_metro or r() > 0.35,
            "median_width_ft":    int(22 + r() * 38),
            "roe_available":      r() > (0.5 if near_metro else 0.25),
        })

    return pd.DataFrame(rows)


# ─────────────────────────────────────────────────────────────────────────────
# Scoring & Filtering
# ─────────────────────────────────────────────────────────────────────────────
def apply_hard_filters(df: pd.DataFrame) -> pd.DataFrame:
    """Drop segments that fail any hard physical requirement."""
    mask = (
        df["has_straight_10mi"].astype(bool) &
        df["has_lane_room"].astype(bool) &
        df["speed_zone_ok"].astype(bool) &
        ~df["env_constraint"].astype(bool) &
        (df["ev_share_pct"] >= THRESHOLDS["min_ev_share_pct"]) &
        (df["interchange_count"] <= THRESHOLDS["max_interchange_count"])
    )
    excluded = (~mask).sum()
    print(f"  Hard filter: {mask.sum()} pass / {excluded} excluded")
    return df[mask].copy()


def score_segments(df: pd.DataFrame) -> pd.DataFrame:
    """Add normalised score columns and a weighted composite score."""
    df = df.copy()

    df["score_ev"]          = (df["ev_share_pct"] / 12).clip(0, 1)
    df["score_power"]       = (1 - df["power_dist_mi"] / 10).clip(0, 1)
    df["score_weather"]     = (1 - df["weather_risk"]).clip(0, 1)
    df["score_incidents"]   = (1 - df["incident_rate"]).clip(0, 1)
    df["score_interchange"] = (1 - df["interchange_count"] / 8).clip(0, 1)

    w = WEIGHTS
    df["composite_score"] = (
        w["ev_traffic"]        * df["score_ev"] +
        w["power_proximity"]   * df["score_power"] +
        w["low_weather_risk"]  * df["score_weather"] +
        w["low_incident_rate"] * df["score_incidents"] +
        w["low_interchange"]   * df["score_interchange"]
    )

    def grade(s):
        if s >= GRADE_THRESHOLDS["A"]: return "A"
        if s >= GRADE_THRESHOLDS["B"]: return "B"
        return "C"

    df["grade"] = df["composite_score"].apply(grade)
    return df


def apply_grade_cutoff(df: pd.DataFrame) -> pd.DataFrame:
    cutoff = THRESHOLDS["grade_cutoff"]
    allowed = list(cutoff)  # "AB" → ["A","B"]
    return df[df["grade"].isin(allowed)].sort_values("composite_score", ascending=False)


# ─────────────────────────────────────────────────────────────────────────────
# Output
# ─────────────────────────────────────────────────────────────────────────────
def print_results(df: pd.DataFrame):
    CORRIDOR_NAMES = {"stl-chi": "STL → Chicago (I-55/I-57)", "stl-kc": "STL → Kansas City (I-70)"}

    print("\n" + "═" * 80)
    print("  EV GUIDED CHARGING LANE — QUALIFYING SITE LOCATIONS")
    print("═" * 80)
    print(f"  Hard requirements: straight 10-mi road | 60-mph zone | lane room | no env constraints")
    print(f"  Min EV share: {THRESHOLDS['min_ev_share_pct']}%  |  Max interchanges: {THRESHOLDS['max_interchange_count']}  |  Grade cutoff: {THRESHOLDS['grade_cutoff']}")
    print(f"  Total qualifying sites: {len(df)}\n")

    for corridor, group in df.groupby("corridor"):
        print(f"  {'─'*70}")
        print(f"  CORRIDOR: {CORRIDOR_NAMES.get(corridor, corridor)}  ({len(group)} sites)")
        print(f"  {'─'*70}")

        for rank, (_, row) in enumerate(group.iterrows(), 1):
            score_pct = f"{row['composite_score']*100:.0f}%"
            print(
                f"  #{rank:<3} {row['segment_id']:<16} "
                f"Grade {row['grade']}  Score {score_pct:<5}  "
                f"Mile {row['mile_marker_start']:>3}–{row['mile_marker_end']:<3}  "
                f"EV {row['ev_share_pct']:>4.1f}%  "
                f"Power {row['power_dist_mi']:>4.1f} mi  "
                f"Interchanges {int(row['interchange_count'])}"
            )
            print(
                f"       WX risk {row['weather_risk']:.2f}  "
                f"Incident rate {row['incident_rate']:.2f}  "
                f"Median {int(row['median_width_ft'])} ft  "
                f"ROE: {'Yes' if row['roe_available'] else 'TBD'}  "
                f"Sub: {row['substation_id'] or 'None nearby'}"
            )
            print(f"       Geometry: {row['geometry_wkt'][:80]}")
            print()

    print("═" * 80)

    # Summary table
    summary = (
        df.groupby(["corridor", "grade"])
          .size()
          .unstack(fill_value=0)
          .rename(index=CORRIDOR_NAMES)
    )
    print("\n  GRADE SUMMARY")
    print(summary.to_string())
    print()


def export_csv(df: pd.DataFrame, path: str = "ev_lane_results.csv"):
    out_cols = [
        "segment_id", "corridor", "route_name",
        "mile_marker_start", "mile_marker_end",
        "grade", "composite_score",
        "ev_share_pct", "aadt", "truck_pct",
        "power_dist_mi", "substation_id",
        "interchange_count", "weather_risk", "incident_rate",
        "median_width_ft", "roe_available",
        "geometry_wkt",
    ]
    available = [c for c in out_cols if c in df.columns]
    df[available].to_csv(path, index=False)
    print(f"  Results exported → {path}")


# ─────────────────────────────────────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────────────────────────────────────
def main():
    print("\nLoading segment data...")

    # ── Swap this block to use Snowflake ──────────────────────────────────────
    # conn = snowflake.connector.connect(...)
    # df = load_from_snowflake(conn)
    # ─────────────────────────────────────────────────────────────────────────
    df = load_synthetic_data()
    print(f"  Loaded {len(df)} segments across {df['corridor'].nunique()} corridors")

    print("Applying hard filters...")
    df = apply_hard_filters(df)

    print("Scoring segments...")
    df = score_segments(df)
    df = apply_grade_cutoff(df)
    print(f"  {len(df)} sites qualify after grade cutoff '{THRESHOLDS['grade_cutoff']}'")

    print_results(df)
    export_csv(df, "ev_lane_results.csv")


if __name__ == "__main__":
    main()


In [ ]:
%%sql -r dataframe_1
SELECT SUM(value)
FROM INTERCHANGES 
WHERE distance > 10 #this is just to calculate the distance
AND (SELECT COUNT(*) FROM INCIDENTS WHERE crashes < 5) > 0 #this could count the crashes
AND (SELECT COUNT(*) FROM SPEED_LIMIT WHERE limit > 70) > 0;


In [ ]:
import csv
import sqlite3  # swap with your actual DB driver

# --- Connect to your database ---
# For SQLite:
conn = sqlite3.connect("your_database.db")

# For other databases, replace with the appropriate driver, e.g.:
# import psycopg2
# conn = psycopg2.connect(host="...", dbname="...", user="...", password="...")

cursor = conn.cursor()

# --- Your query ---
query = """
SELECT SUM(value)
FROM INTERCHANGES
WHERE distance > 10
  AND (SELECT COUNT(*) FROM INCIDENTS WHERE crashes < 5) > 0
  AND (SELECT COUNT(*) FROM SPEED_LIMIT WHERE limit > 70) > 0
"""

cursor.execute(query)
rows = cursor.fetchall()

# --- Write to CSV ---
output_file = "output.csv"

with open(output_file, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["sum_of_value"])  # header
    writer.writerows(rows)

print(f"Results exported to {output_file}")

conn.close()

In [ ]:
"""
EV Guided Charging Lane — CSV Export Pipeline
Runs the full analysis (load → filter → score → grade) and writes results to CSV.
"""

import sys

try:
    import pandas as pd
except ImportError:
    print("pandas not found. Install with: pip install pandas")
    sys.exit(1)

import random
from datetime import datetime

# ── Configuration ─────────────────────────────────────────────────────────────
WEIGHTS = {
    "ev_traffic":        0.25,
    "power_proximity":   0.25,
    "low_weather_risk":  0.20,
    "low_incident_rate": 0.15,
    "low_interchange":   0.15,
}

THRESHOLDS = {
    "min_ev_share_pct":      4.0,
    "max_interchange_count": 4,
    "max_power_dist_mi":     5.0,
    "grade_cutoff":          "AB",
}

GRADE_THRESHOLDS = {"A": 0.68, "B": 0.50}

OUTPUT_FILE = f"ev_lane_results_{datetime.today().strftime('%Y%m%d')}.csv"


# ── Step 1: Load Data ─────────────────────────────────────────────────────────
def load_synthetic_data() -> pd.DataFrame:
    random.seed(42)
    def r(): return random.random()
    rows = []

    for i in range(55):
        near_metro   = i < 5 or i > 48
        in_northern  = i > 35
        rows.append({
            "segment_id":         f"STL-CHI-{i+1:03d}",
            "corridor":           "stl-chi",
            "route_name":         "I-55 / I-57",
            "mile_marker_start":  10 + i * 5,
            "mile_marker_end":    20 + i * 5,
            "geometry_wkt":       f"LINESTRING(-90.19 38.63, {-90.19+(i/54)*2.56:.4f} {38.63+(i/54)*3.25:.4f})",
            "aadt":               int(18000 + r() * 32000),
            "ev_share_pct":       round(2 + r() * 10, 1),
            "truck_pct":          round(15 + r() * 20, 1),
            "power_dist_mi":      round(0.2 + r() * 8, 1),
            "substation_id":      f"SUB-{int(r()*30+1):03d}" if r() > 0.5 else None,
            "interchange_count":  int(3 + r() * 5) if near_metro else int(r() * 4),
            "env_constraint":     r() < (0.22 if 30 < i < 50 else 0.12),
            "weather_risk":       round(0.4 + r() * 0.5, 2) if in_northern else round(0.1 + r() * 0.35, 2),
            "incident_rate":      round(0.45 + r() * 0.5, 2) if near_metro else round(0.05 + r() * 0.35, 2),
            "has_straight_10mi":  r() > (0.6 if near_metro else 0.3),
            "has_lane_room":      r() > (0.55 if near_metro else 0.25),
            "speed_zone_ok":      not near_metro or r() > 0.4,
            "median_width_ft":    int(20 + r() * 40),
            "roe_available":      r() > (0.55 if near_metro else 0.3),
        })

    for i in range(48):
        near_metro = i < 5 or i > 42
        rows.append({
            "segment_id":         f"STL-KC-{i+1:03d}",
            "corridor":           "stl-kc",
            "route_name":         "I-70",
            "mile_marker_start":  10 + i * 5,
            "mile_marker_end":    20 + i * 5,
            "geometry_wkt":       f"LINESTRING(-90.19 38.63, {-90.19-(i/47)*4.39:.4f} {38.63+(i/47)*0.47:.4f})",
            "aadt":               int(15000 + r() * 28000),
            "ev_share_pct":       round(2 + r() * 10, 1),
            "truck_pct":          round(18 + r() * 17, 1),
            "power_dist_mi":      round(0.3 + r() * 9, 1),
            "substation_id":      f"SUB-{int(r()*30+1):03d}" if r() > 0.45 else None,
            "interchange_count":  int(3 + r() * 5) if near_metro else int(r() * 3),
            "env_constraint":     r() < (0.18 if 15 < i < 30 else 0.1),
            "weather_risk":       round(0.08 + r() * 0.32, 2),
            "incident_rate":      round(0.4 + r() * 0.5, 2) if near_metro else round(0.05 + r() * 0.3, 2),
            "has_straight_10mi":  r() > (0.55 if near_metro else 0.2),
            "has_lane_room":      r() > (0.5 if near_metro else 0.2),
            "speed_zone_ok":      not near_metro or r() > 0.35,
            "median_width_ft":    int(22 + r() * 38),
            "roe_available":      r() > (0.5 if near_metro else 0.25),
        })

    return pd.DataFrame(rows)


# ── Optional: Snowflake loader (uncomment + fill credentials to use) ──────────
# def load_from_snowflake() -> pd.DataFrame:
#     import snowflake.connector
#     conn = snowflake.connector.connect(
#         user='YOUR_USER', password='YOUR_PASSWORD',
#         account='YOUR_ACCOUNT', warehouse='YOUR_WAREHOUSE',
#         database='YOUR_DATABASE', schema='YOUR_SCHEMA',
#     )
#     query = "SELECT ... FROM ROAD_SEGMENTS JOIN ..."  # full join query here
#     cursor = conn.cursor()
#     cursor.execute(query)
#     df = cursor.fetch_pandas_all()
#     df.columns = [c.lower() for c in df.columns]
#     return df


# ── Step 2: Hard Filters ──────────────────────────────────────────────────────
def apply_hard_filters(df: pd.DataFrame) -> pd.DataFrame:
    mask = (
        df["has_straight_10mi"].astype(bool) &
        df["has_lane_room"].astype(bool) &
        df["speed_zone_ok"].astype(bool) &
        ~df["env_constraint"].astype(bool) &
        (df["ev_share_pct"] >= THRESHOLDS["min_ev_share_pct"]) &
        (df["interchange_count"] <= THRESHOLDS["max_interchange_count"])
    )
    print(f"  Hard filter: {mask.sum()} pass / {(~mask).sum()} excluded")
    return df[mask].copy()


# ── Step 3: Score ─────────────────────────────────────────────────────────────
def score_segments(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["score_ev"]          = (df["ev_share_pct"] / 12).clip(0, 1)
    df["score_power"]       = (1 - df["power_dist_mi"] / 10).clip(0, 1)
    df["score_weather"]     = (1 - df["weather_risk"]).clip(0, 1)
    df["score_incidents"]   = (1 - df["incident_rate"]).clip(0, 1)
    df["score_interchange"] = (1 - df["interchange_count"] / 8).clip(0, 1)

    w = WEIGHTS
    df["composite_score"] = (
        w["ev_traffic"]        * df["score_ev"] +
        w["power_proximity"]   * df["score_power"] +
        w["low_weather_risk"]  * df["score_weather"] +
        w["low_incident_rate"] * df["score_incidents"] +
        w["low_interchange"]   * df["score_interchange"]
    ).round(4)

    def grade(s):
        if s >= GRADE_THRESHOLDS["A"]: return "A"
        if s >= GRADE_THRESHOLDS["B"]: return "B"
        return "C"

    df["grade"] = df["composite_score"].apply(grade)
    return df


# ── Step 4: Apply Grade Cutoff ────────────────────────────────────────────────
def apply_grade_cutoff(df: pd.DataFrame) -> pd.DataFrame:
    allowed = list(THRESHOLDS["grade_cutoff"])
    return df[df["grade"].isin(allowed)].sort_values("composite_score", ascending=False)


# ── Step 5: Export to CSV ─────────────────────────────────────────────────────
def export_csv(df: pd.DataFrame, path: str):
    out_cols = [
        "segment_id", "corridor", "route_name",
        "mile_marker_start", "mile_marker_end",
        "grade", "composite_score",
        "score_ev", "score_power", "score_weather", "score_incidents", "score_interchange",
        "ev_share_pct", "aadt", "truck_pct",
        "power_dist_mi", "substation_id",
        "interchange_count", "weather_risk", "incident_rate",
        "median_width_ft", "roe_available",
        "geometry_wkt",
    ]
    available = [c for c in out_cols if c in df.columns]
    df[available].to_csv(path, index=False)
    print(f"  Exported {len(df)} qualifying sites → {path}")


# ── Main Pipeline ─────────────────────────────────────────────────────────────
def run_pipeline():
    print("\n[1/5] Loading segment data...")
    df = load_synthetic_data()
    # df = load_from_snowflake()  # ← swap here for real Snowflake data
    print(f"  Loaded {len(df)} segments across {df['corridor'].nunique()} corridors")

    print("[2/5] Applying hard filters...")
    df = apply_hard_filters(df)

    print("[3/5] Scoring segments...")
    df = score_segments(df)

    print("[4/5] Applying grade cutoff...")
    df = apply_grade_cutoff(df)
    print(f"  {len(df)} sites qualify at grade cutoff '{THRESHOLDS['grade_cutoff']}'")

    print("[5/5] Exporting to CSV...")
    export_csv(df, OUTPUT_FILE)

    print(f"\nDone. Output: {OUTPUT_FILE}\n")
    return df


if __name__ == "__main__":
    run_pipeline()

In [ ]:
%%sql -r dataframe_2
-- ============================================================
-- EV GUIDED CHARGING LANE — SITE SELECTION
-- Corridors: STL → Chicago (I-55/I-57) | STL → Kansas City (I-70)
-- ============================================================

SELECT
    rs.segment_id,
    rs.corridor,
    rs.route_name,
    rs.mile_marker_start,
    rs.mile_marker_end,

    -- Traffic
    tc.aadt,
    tc.ev_share_pct,

    -- Power infrastructure
    ROUND(MIN(ST_DISTANCE(rs.geog, pi.geog) / 1609.34), 1) AS power_dist_mi,

    -- Interchange count within segment
    COUNT(DISTINCT ic.interchange_id)                       AS interchange_count,

    -- Weather & incident risk
    wr.risk_score       AS weather_risk,
    inc.crash_rate_normalized AS incident_rate,

    -- Physical flags
    rs.median_width_ft,
    rs.roe_available

FROM ROAD_SEGMENTS rs

-- Traffic data
JOIN TRAFFIC_COUNTS tc
    ON rs.segment_id = tc.segment_id

-- Nearest power source (substation or transmission line)
JOIN POWER_INFRA pi
    ON ST_DISTANCE(rs.geog, pi.geog) / 1609.34 <= 5.0   -- within 5 miles

-- Interchanges within the segment
LEFT JOIN INTERCHANGES ic
    ON ic.segment_id = rs.segment_id

-- Weather risk
JOIN WEATHER_RISK wr
    ON rs.segment_id = wr.segment_id

-- Incident / crash rate
JOIN INCIDENTS inc
    ON rs.segment_id = inc.segment_id

-- ── Hard Requirements ────────────────────────────────────────
WHERE
    -- Correct corridors only
    rs.corridor IN ('stl-chi', 'stl-kc')

    -- Room for a new dedicated lane
    AND rs.has_lane_room = TRUE
    AND rs.median_width_ft >= 20

    -- Speed zone supports 60 mph → 30 mph transition
    AND rs.speed_zone_ok = TRUE

    -- At least 10 miles of open, straight road
    AND rs.has_straight_10mi = TRUE

    -- Right-of-way available
    AND rs.roe_available = TRUE

    -- No wetlands or protected areas overlap
    AND rs.segment_id NOT IN (
        SELECT DISTINCT rs2.segment_id
        FROM ROAD_SEGMENTS rs2
        JOIN ENV_CONSTRAINTS ec
            ON ST_INTERSECTS(rs2.geog, ec.geog)
    )

    -- Enough EV traffic to justify infrastructure
    AND tc.ev_share_pct >= 4.0

GROUP BY
    rs.segment_id, rs.corridor, rs.route_name,
    rs.mile_marker_start, rs.mile_marker_end,
    tc.aadt, tc.ev_share_pct,
    wr.risk_score, inc.crash_rate_normalized,
    rs.median_width_ft, rs.roe_available

-- ── Soft Filters (quality thresholds) ───────────────────────
HAVING
    COUNT(DISTINCT ic.interchange_id) <= 4   -- few interchanges = easier entry/exit
    AND MIN(ST_DISTANCE(rs.geog, pi.geog) / 1609.34) <= 5.0

-- ── Ranking: best sites first ────────────────────────────────
ORDER BY
    (tc.ev_share_pct * 0.25)                                          -- high EV traffic
    + ((1 - MIN(ST_DISTANCE(rs.geog, pi.geog) / 1609.34) / 10) * 0.25) -- close to power
    + ((1 - wr.risk_score) * 0.20)                                    -- low weather risk
    + ((1 - inc.crash_rate_normalized) * 0.15)                        -- low incident rate
    + ((1 - COUNT(DISTINCT ic.interchange_id) / 8.0) * 0.15)         -- few interchanges
    DESC;